In [1]:

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from openai import OpenAI

DATA_PATH = "datset.csv"   
OPENAI_MODEL = "gpt-5-mini"   # lower-cost LLM for feature engineering
DISTILBERT_MODEL = "distilbert-base-uncased"
MAX_LEN = 256
LOOKBACK_TXNS = 10
USE_LLM = True
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
import os


# client = OpenAI()


/data1/rachit/.conda/envs/cond/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH="../../transactions/card_transaction.v1.csv"
df = pd.read_csv(DATA_PATH)

In [3]:

df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

df["Errors?"] = (df["Errors?"] != "None").astype(int)

df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

df["Hour"] = (df["Time"] // 3600) % 24
df["Is Fraud?"] = df["Is Fraud?"].map({"Yes": 1, "No": 0})

df = df.fillna(0)

# print(df.dtypes)
# print(df.head())

/tmp/ipykernel_791565/3282332532.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour


In [4]:
df["Merchant City"] = df["Merchant City"].astype(str)
df["Merchant State"] = df["Merchant State"].astype(str)

In [5]:
print(df.dtypes)
print(df.head())

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time              float64
Amount            float64
Use Chip          float64
Merchant Name       int64
Merchant City      object
Merchant State     object
Zip               float64
MCC                 int64
Errors?             int64
Is Fraud?           int64
Hour              float64
dtype: object
   User  Card  Year  Month  Day  Time  Amount  Use Chip        Merchant Name  \
0     0     0  2002      9    1   0.0  134.09       0.0  3527213246127876953   
1     0     0  2002      9    1   0.0   38.48       0.0  -727612092139916043   
2     0     0  2002      9    2   0.0  120.34       0.0  -727612092139916043   
3     0     0  2002      9    2   0.0  128.95       0.0  3414527459579106770   
4     0     0  2002      9    3   0.0  104.71       0.0  5817218446178736267   

   Merchant City Merchant State      Zip   MCC  Errors?  Is Fraud?  Hour  
0       La Ver

## Run form here

In [139]:
# STEP 4: Connect to Ollama
import requests

def call_llm(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()
    
    if "response" in data:
        return data["response"]
    else:
        print("Error:", data)
        return None

In [140]:

# from dotenv import load_dotenv
# import os
# import google.generativeai as genai

# load_dotenv()

# genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [141]:
# for m in genai.list_models():
#     print(m.name, "->", m.supported_generation_methods)

In [142]:
# model = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [143]:

# df["Amount"] = df["Amount"].replace('[\$,]', '', regex=True).astype(float)

# df["Hour"] = pd.to_datetime(df["Time"], errors="coerce").dt.hour

# df["Use Chip"] = df["Use Chip"].map({"Yes": 1, "No": 0}).fillna(0)

# df["Errors?"] = (df["Errors?"] != "None").astype(int)

# df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")

# df = df.fillna(0)

In [144]:
# PROMPT = """
# You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds —  directly)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded —  directly)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a Classifier detect fraud.

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users


# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features

# IMPORTANT RULES:

# - Always use "Hour" for time-based features (NOT Time)
# - Merchant City / State → ONLY nunique or count
# - Ensure all outputs are numeric scalars per user

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# Generate 12–15 high-quality fraud detection features.

# Focus on:
# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior
# - error behavior
# - chip usage patterns

# Do NOT compute values.
# Do NOT output anything outside JSON.


# """

In [145]:
PROMPT = """
You are a fraud analytics feature engineering assistant.

The features you generate will be used to train a RandomForestClassifier.

==============================
DATASET SCHEMA (STRICT)
==============================

- User → int64 (group key)
- Card → int64 
- Year → int64
- Month → int64
- Day → int64
- Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
- Hour → float64 (0–23, derived from Time — USE THIS)
- Amount → float64 (numeric)
- Use Chip → float64 (1 = Yes, 0 = No)
- Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
- Merchant City → object (categorical — ONLY use nunique/count)
- Merchant State → object (categorical — ONLY use nunique/count)
- Zip → float64 (numeric)
- MCC → int64 (category code — can use nunique/count)
- Errors? → int64 (1 = error, 0 = no error)
- Is Fraud? → int64 (target label — DO NOT USE)

==============================
YOUR TASK
==============================

Design USER-LEVEL features for fraud detection.

Each feature MUST produce ONE value per User.

==============================
STRICT REQUIREMENTS (VERY IMPORTANT)
==============================

- Features must be NUMERIC only (int or float)
- Must be directly usable in sklearn (no preprocessing needed)
- MUST be computed ONLY from df
- MUST NOT depend on other generated features
- SAME features for ALL users

FEATURE DESIGN FOCUS:

- spending patterns
- transaction frequency
- merchant/category diversity
- geographic spread
- time behavior (USE Hour only)
- error behavior
- chip usage patterns

==============================
CRITICAL RULE (VERY IMPORTANT)
==============================

- Each feature must apply ONLY ONE aggregation
- After df.groupby('User'), apply EXACTLY ONE aggregation
- DO NOT apply aggregation on top of aggregation

INVALID:
df.groupby('User')['Amount'].mean().mean()
df.groupby('User')['MCC'].nunique().mean()
df.groupby('User').size().mean()

VALID:
df.groupby('User')['Amount'].mean()
df.groupby('User')['MCC'].nunique()
df.groupby('User').size()

If the result is a single number (scalar), the feature is INVALID.

==============================
CRITICAL EXECUTION RULES (MANDATORY)
==============================

Each feature MUST follow EXACTLY this pattern:

df.groupby('User')['COLUMN'].AGG()

OR:

df.groupby('User').size()

Where:
- COLUMN = exactly ONE column
- AGG MUST be one of: mean, sum, count, nunique, max, min, std

==============================
ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
==============================

- NEVER use df['User'].map()
- NEVER use .values
- NEVER use reset_index()
- NEVER use multiple columns (NO [['col1','col2']])
- NEVER use resample()
- NEVER use datetime operations
- NEVER return DataFrame (must return Series)
- NEVER perform arithmetic between multiple aggregations
- NEVER nest groupby
- NEVER use apply()
- NEVER chain column selection like: df.groupby(... )['col1']['col2']

==============================
VALID EXAMPLES (ONLY THESE TYPES)
==============================

df.groupby('User')['Amount'].mean()

df.groupby('User')['MCC'].nunique()

df.groupby('User')['Errors?'].sum()

df.groupby('User')['Hour'].std()

df.groupby('User').size()

==============================
INVALID EXAMPLES (DO NOT GENERATE)
==============================

df['User'].map(...)

df.groupby('User')[['Amount','Hour']]

df.groupby('User').resample('H')

df.groupby('User')['Amount'].mean().reset_index()

df.groupby('User')['Amount'].mean() / df.groupby('User').size()

==============================
FEATURE DESIGN FOCUS
==============================

- spending patterns (mean, std, max)
- transaction frequency (counts)
- merchant/category diversity (nunique)
- geographic spread (city/state/zip diversity)
- time behavior (Hour-based stats)
- error behavior
- chip usage patterns

==============================
OUTPUT FORMAT (STRICT JSON)
==============================

[
  {
    "feature_name": "...",
    "pandas_code": "...",
    "description": "...",
    "type": "numeric"
  }
]

==============================
FINAL INSTRUCTIONS
==============================

- Generate 12–15 high-quality features
- Each feature MUST return a pandas Series indexed by User
- Length MUST equal number of unique users
- Do NOT compute values
- Do NOT explain anything
- Output ONLY JSON
- Output MUST start with '[' and end with ']'
- Use double quotes only
"""

In [146]:
# PROMPT="""You are a fraud analytics feature engineering assistant.

# The features you generate will be used to train a RandomForestClassifier.

# Dataset columns and TYPES (STRICT — DO NOT ASSUME ANYTHING ELSE):

# - User → int64 (group key)
# - Card → int64 
# - Year → int64
# - Month → int64
# - Day → int64
# - Time → float64 (transaction timestamp in seconds — DO NOT USE DIRECTLY)
# - Hour → float64 (0–23, derived from Time — USE THIS)
# - Amount → float64 (numeric)
# - Use Chip → float64 (1 = Yes, 0 = No)
# - Merchant Name → int64 (encoded — DO NOT USE DIRECTLY)
# - Merchant City → object (categorical — ONLY use nunique/count)
# - Merchant State → object (categorical — ONLY use nunique/count)
# - Zip → float64 (numeric)
# - MCC → int64 (category code — can use nunique/count)
# - Errors? → int64 (1 = error, 0 = no error)
# - Is Fraud? → int64 (target label — DO NOT USE)

# ---

# YOUR TASK:

# Design GLOBAL USER-LEVEL FEATURES and behavioral patterns that can help a RandomForestClassifier detect fraud.

# ---

# STRICT REQUIREMENTS (VERY IMPORTANT):

# - Features must be NUMERIC only (int or float)
# - Must be directly usable in sklearn (no preprocessing needed)
# - Must use pandas groupby('User')
# - MUST be computed ONLY from df (no dependency on other generated features)
# - SAME features for ALL users

# ---

# FEATURE DESIGN FOCUS:

# - spending patterns
# - transaction frequency
# - merchant/category diversity
# - geographic spread
# - time behavior (USE Hour only)
# - error behavior
# - chip usage patterns

# ---

# AVOID:

# - raw text usage
# - using Merchant Name directly
# - using Is Fraud?
# - string operations
# - referencing other generated features
# - using Time column
# - non-numeric outputs

# ---

# ==============================
# STRICT EXECUTION TEMPLATE (MANDATORY)
# ==============================

# Every feature MUST follow EXACTLY ONE of these two forms:

# FORM 1:
# df['User'].map(df.groupby('User')['COLUMN'].AGG())

# FORM 2:
# df['User'].map(df.groupby('User').size())

# Where:
# - COLUMN = exactly ONE column name (string)
# - AGG MUST be one of:
#   mean, sum, count, nunique, max, min, std

# ==============================
# ABSOLUTE RESTRICTIONS (NO EXCEPTIONS)
# ==============================

# - NEVER use .values
# - NEVER use reset_index()
# - NEVER use multiple columns (NO [['col1','col2']])
# - NEVER use resample()
# - NEVER use datetime operations
# - NEVER return DataFrame
# - NEVER perform arithmetic between two map() calls
# - NEVER nest groupby operations
# - NEVER use apply()
# - NEVER chain column selection like:
#   df.groupby(... )['col1']['col2']

# - You may ONLY select ONE column after groupby:
#   df.groupby('User')['COLUMN']

# ==============================
# VALID EXAMPLES (ONLY THESE TYPES)
# ==============================

# df['User'].map(df.groupby('User')['Amount'].mean())

# df['User'].map(df.groupby('User')['MCC'].nunique())

# df['User'].map(df.groupby('User')['Errors?'].sum())

# df['User'].map(df.groupby('User').size())

# ==============================
# INVALID EXAMPLES (DO NOT GENERATE)
# ==============================

# df.groupby('User').size().values

# df.groupby('User')[['Amount','Hour']]

# df.groupby('User').resample('H')

# df['User'].map(A) / df['User'].map(B)

# df.groupby('User')['Amount'].mean().reset_index()



# EXAMPLE (FOLLOW THIS EXACT STYLE):

# {
#   "feature_name": "avg_amount_per_user",
#   "pandas_code": "df['User'].map(df.groupby('User')['Amount'].mean())",
#   "description": "Average transaction amount per user",
#   "type": "numeric"
# }

# ---

# OUTPUT FORMAT (STRICT JSON):

# [
#   {
#     "feature_name": "...",
#     "pandas_code": "...",
#     "description": "...",
#     "type": "numeric"
#   }
# ]

# ---

# FINAL INSTRUCTIONS:

# - Generate 12–15 high-quality fraud detection features
# - Do NOT compute values
# - Do NOT explain anything
# - Do NOT output anything outside JSON
# - Ensure VALID JSON (double quotes only)
# ==============================
# FINAL REQUIREMENT
# ==============================

# If a feature cannot be written EXACTLY in one of the VALID forms above,
# DO NOT include it.
# """

In [147]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")

In [148]:
response = call_llm(PROMPT)
print("RAW OUTPUT:\n", response)

RAW OUTPUT:
 Here are the features I've generated for fraud detection:

[
  {
    "feature_name": "mean_amount",
    "pandas_code": "df.groupby('User')['Amount'].mean()",
    "description": "Mean transaction amount per user",
    "type": "numeric"
  },
  {
    "feature_name": "std_amount",
    "pandas_code": "df.groupby('User')['Amount'].std()",
    "description": "Standard deviation of transaction amounts per user",
    "type": "numeric"
  },
  {
    "feature_name": "max_hour",
    "pandas_code": "df.groupby('User')['Hour'].max()",
    "description": "Maximum hour of transactions per user",
    "type": "numeric"
  },
  {
    "feature_name": "mean_hour",
    "pandas_code": "df.groupby('User')['Hour'].mean()",
    "description": "Mean hour of transactions per user",
    "type": "numeric"
  },
  {
    "feature_name": "nunique_merchant_city",
    "pandas_code": "df.groupby('User')['Merchant City'].nunique()",
    "description": "Number of unique merchant cities per user",
    "type": "num

In [149]:
import re

def fix_code(code):
    # Fix missing quotes
    replacements = {
        "[Amount]": "['Amount']",
        "[Time]": "['Time']",
        "[Is Fraud?]": "['Is Fraud?']"
    }
    for k, v in replacements.items():
        code = code.replace(k, v)

    # 🔥 Fix tuple column selection → list
    code = re.sub(
        r"\[['\"](\w+)['\"],\s*['\"](\w+)['\"]\]",
        r"[[ '\1', '\2' ]]",
        code
    )

    # Also fix ('A','B') → ['A','B']
    code = re.sub(
        r"\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)",
        r"['\1', '\2']",
        code
    )

    return code

In [150]:
def extract_json(llm_output):
    try:
        json_str = re.search(r'\[.*\]', llm_output, re.DOTALL).group(0)
        return json.loads(json_str)
    except:
        print("Failed to parse JSON")
        print(llm_output)
        return []
    


In [151]:
# response = model_llm.generate_content(PROMPT)
response=fix_code(response)
features = extract_json(response)



user_df = pd.DataFrame()

user_df = pd.DataFrame(index=df['User'].unique())

for f in features:
    try:
        print("Applying:", f["feature_name"])
        
        result = eval(f["pandas_code"])
        
        # ---- VALIDATION ----
        if not isinstance(result, pd.Series):
            raise ValueError("Not a pandas Series")
        
        # ---- FORCE ALIGNMENT ----
        result = result.reindex(user_df.index)
        
        user_df[f["feature_name"]] = result
        
    except Exception as e:
        print("Error in", f["feature_name"], ":", e)
        

user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()
user_df["is_fraud_user"] = user_df["is_fraud_user"].reindex(user_df.index)

user_df = user_df.fillna(0)

# for f in features:
#     try:
#         print("Applying:", f["feature_name"])
#         user_df[f["feature_name"]] = eval(f["pandas_code"])
#     except Exception as e:
#         print("Error in", f["feature_name"], ":", e)

# user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

# user_df = user_df.fillna(0)



Applying: mean_amount
Applying: std_amount
Applying: max_hour
Applying: mean_hour
Applying: nunique_merchant_city
Applying: nunique_merchant_state
Applying: chip_usage_rate
Applying: error_count
Applying: transaction_count
Error in transaction_count : Not a pandas Series
Applying: min_zip
Applying: max_amount


In [152]:
print("\nFinal Shape:", user_df.shape)
print(user_df.head())


Final Shape: (2000, 11)
   mean_amount  std_amount  max_hour  mean_hour  nunique_merchant_city  \
0    81.299989   94.159093       0.0        0.0                    295   
1    81.118050  156.784691       0.0        0.0                    263   
2    35.159687   54.298552       0.0        0.0                    314   
3   117.277603  268.654472       0.0        0.0                    392   
4    97.011698  138.619564       0.0        0.0                    402   

   nunique_merchant_state  chip_usage_rate  error_count  min_zip  max_amount  \
0                      41              0.0        19963      0.0     1409.40   
1                      48              0.0         8919      0.0     3750.60   
2                      41              0.0        41978      0.0     1195.57   
3                      67              0.0        10117      0.0     6820.20   
4                      49              0.0        18542      0.0     3613.22   

   is_fraud_user  
0              1  
1          

In [153]:
user_df["is_fraud_user"] = df.groupby("User")["Is Fraud?"].max()

In [154]:
print(user_df.shape)

(2000, 11)


In [155]:
# Separate label first
y = user_df["is_fraud_user"]

# Remove non-numeric features
X = user_df.drop(columns=["is_fraud_user"])
X = X.select_dtypes(include=["number"])

In [156]:
print(y.isna().sum())

0


In [157]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

In [158]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"   
)

model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [159]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.88      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.84       400
weighted avg       0.87      0.87      0.87       400

ROC-AUC: 0.9015579329719913


In [160]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)

error_count               0.307002
nunique_merchant_city     0.218572
nunique_merchant_state    0.207458
max_amount                0.131601
std_amount                0.071448
mean_amount               0.063808
min_zip                   0.000112
mean_hour                 0.000000
max_hour                  0.000000
chip_usage_rate           0.000000
dtype: float64


In [161]:
feature_names = [f["feature_name"] for f in features]

In [162]:
top_features = importance.to_dict()

In [163]:
report1 = classification_report(y_test, y_pred)
roc1 = roc_auc_score(y_test, y_prob)

llm_input = {
    "existing_features": feature_names,
    "classification_report": report1,
    "roc_auc": roc1,
    "top_features": top_features
}


In [164]:
print(report1)
print(importance)
print(roc1)

              precision    recall  f1-score   support

           0       0.88      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.84       400
weighted avg       0.87      0.87      0.87       400

error_count               0.307002
nunique_merchant_city     0.218572
nunique_merchant_state    0.207458
max_amount                0.131601
std_amount                0.071448
mean_amount               0.063808
min_zip                   0.000112
mean_hour                 0.000000
max_hour                  0.000000
chip_usage_rate           0.000000
dtype: float64
0.9015579329719913


In [165]:
def get_prompt(llm_input):
  PROMPT = f"""
  You are a fraud analytics feature engineering expert.

  A RandomForestClassifier was trained on USER-LEVEL features.

  ==============================
  DATASET SCHEMA (STRICT)
  ==============================

  - User → int64 (group key)
  - Card → int64 (DO NOT USE)
  - Year → int64
  - Month → int64
  - Day → int64
  - Time → float64 (timestamp in seconds — DO NOT USE directly)
  - Hour → float64 (0–23 — USE THIS for time features)
  - Amount → float64
  - Use Chip → float64 (1 = Yes, 0 = No)
  - Merchant Name → int64 (DO NOT USE directly)
  - Merchant City → object (ONLY use nunique/count)
  - Merchant State → object (ONLY use nunique/count)
  - Zip → float64
  - MCC → int64
  - Errors? → int64 (1 = error, 0 = no error)
  - Is Fraud? → int64 (TARGET — DO NOT USE)

  ==============================
  CURRENT FEATURES
  ==============================
  {llm_input["existing_features"]}

  ==============================
  MODEL PERFORMANCE
  ==============================

  Classification Report:
  {llm_input["classification_report"]}

  ROC-AUC:
  {llm_input["roc_auc"]}

  Top Important Features:
  {llm_input["top_features"]}

  ==============================
  YOUR TASK
  ==============================

  1. Analyze weaknesses in the model (especially fraud recall)
  2. Identify missing behavioral signals
  3. Suggest 8–12 NEW or IMPROVED features

  ==============================
  STRICT RULES (VERY IMPORTANT)
  ==============================

  - Features must be NUMERIC only
  - Must work directly in sklearn (no preprocessing)
  - Must use pandas groupby('User')
  - MUST be directly computable from df
  - MUST NOT depend on other generated features
  - MUST NOT duplicate existing features


  AVOID:
  - raw text usage
  - using Merchant Name directly
  - using Is Fraud?
  - referencing other generated features
  - using Time (use Hour only)

  IMPORTANT:
  - Each pandas_code MUST return a pandas Series indexed by User

  ==============================
  OUTPUT FORMAT (STRICT JSON)
  ==============================

  [
    {{
      "feature_name": "...",
      "pandas_code": "...",
      "reason": "..."
    }}
  ]

  Focus on:
  - behavioral anomalies
  - spending variability
  - temporal patterns (Hour-based)
  - transaction bursts
  - merchant/category diversity
  - geographic spread
  - error behavior
  - chip usage changes

  Do NOT compute values.
  Do NOT output anything outside JSON.
  """
  return PROMPT

In [166]:
# model_llm = genai.GenerativeModel("models/gemini-flash-lite-latest")
# response = model_llm.generate_content(PROMPT)


In [167]:
# a=1
# for a in range(1,11):
#     PROMPT=get_prompt(llm_input)
#     response = call_llm(PROMPT)

#     output = response.text.replace("```json", "").replace("```", "").strip()

#     new_features = json.loads(output)
#     base_features = []

#     for f in features:
#         code = f["pandas_code"]
        
#         # Only keep df-based features
#         if "df.groupby" in code and "user_" not in code.split("=")[-1]:
#             base_features.append(f)
            
#     for f in base_features:
#         try:
#             user_df[f["feature_name"]] = eval(f["pandas_code"])
#         except Exception as e:
#             print("Base error:", f["feature_name"], e)
    
            
#     a+=1

In [168]:
base_features = []

for f in features:
    code = f["pandas_code"]
    
    # Only keep df-based features
    if "df.groupby" in code and "user_" not in code.split("=")[-1]:
        base_features.append(f)

In [169]:
for f in base_features:
    try:
        user_df[f["feature_name"]] = eval(f["pandas_code"])
    except Exception as e:
        print("Base error:", f["feature_name"], e)

In [170]:
X = user_df.drop(columns=["is_fraud_user"])
y = user_df["is_fraud_user"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [171]:

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ================================
# 🔹 11. EVALUATE
# ================================
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== ROC AUC ===")
print(roc_auc_score(y_test, y_prob))

# ================================
# 🔹 12. FEATURE IMPORTANCE
# ================================
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n=== Top Features ===")
print(importance.head(10))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.91      0.69      0.79       131
           1       0.87      0.97      0.91       269

    accuracy                           0.88       400
   macro avg       0.89      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400


=== ROC AUC ===
0.9023808848151196

=== Top Features ===
error_count               0.316429
nunique_merchant_city     0.216502
nunique_merchant_state    0.184737
max_amount                0.128333
std_amount                0.081200
mean_amount               0.072605
min_zip                   0.000194
max_hour                  0.000000
mean_hour                 0.000000
chip_usage_rate           0.000000
dtype: float64


In [172]:
report2 = classification_report(y_test, y_pred)
roc2 = roc_auc_score(y_test, y_prob)

In [173]:
# precision    recall  f1-score   support

#           No       0.87      0.72      0.79       131
#          Yes       0.87      0.95      0.91       269

#     accuracy                           0.87       400
#    macro avg       0.87      0.83      0.85       400
# weighted avg       0.87      0.87      0.87       400

# ROC-AUC: 0.9174210391895343

In [174]:
print(report1)
print(report2)


              precision    recall  f1-score   support

           0       0.88      0.69      0.78       131
           1       0.87      0.96      0.91       269

    accuracy                           0.87       400
   macro avg       0.87      0.83      0.84       400
weighted avg       0.87      0.87      0.87       400

              precision    recall  f1-score   support

           0       0.91      0.69      0.79       131
           1       0.87      0.97      0.91       269

    accuracy                           0.88       400
   macro avg       0.89      0.83      0.85       400
weighted avg       0.88      0.88      0.87       400



In [175]:
print(roc1)
print(roc2)

0.9015579329719913
0.9023808848151196
